# A presenter for *Who is calling this server?*

This notebook turns the course narration into a photoreal talking presenter, one clip per episode, using free Kaggle GPU time and open models: a synthetic portrait from **SDXL-Turbo**, animated by **SadTalker** (Apache-2.0), faces sharpened by **GFPGAN**.

Nothing here is a real person: the presenter is generated, so there is no likeness to license.

## Before you press Run all

1. **Settings → Accelerator: GPU T4 x2.** Settings → Internet: **On**. (GPU needs a phone-verified Kaggle account.)
2. **Add data →** upload `presenter-input.zip` as a new Dataset (any name). It holds the 14 narration files and a manifest. Kaggle unzips it for you; the notebook finds it either way.
3. Leave `EPISODES = ["01-no-authentication"]` for the first run — it renders the shortest episode in ~10 minutes so you can check the presenter before spending hours. Then set it to `"all"` and run again; finished episodes are skipped.

**Time:** roughly 2–4 hours for the whole course with the face enhancer (both GPUs are used), well inside a 12-hour session and the 30 h/week quota. **Output:** `presenter-clips.zip` in the notebook's Output tab — download it, unzip into `video/presenter/` in the repository, run `npm run video`.

If the session dies, run again: episodes already in `/kaggle/working/presenter/` are not redone.

In [ ]:
# ── what to render ──────────────────────────────────────────────────────────────────────
EPISODES = ["01-no-authentication"]   # a list of slugs, or "all"

# ── the presenter ───────────────────────────────────────────────────────────────────────
PRESENTER_IMAGE = None      # a path to your own portrait (frontal, one face, plain background) — or None to generate one
PRESENTER_PICK = 0          # which of the generated candidates to use (0–3); the grid is shown below
PRESENTER_PROMPT = (
    "studio portrait photograph of a television news presenter, centered, head and shoulders, "
    "looking straight into the camera, calm friendly expression, mouth closed, plain dark grey background, "
    "soft even studio light, 85mm lens, sharp focus, photorealistic"
)
PRESENTER_NEGATIVE = "cartoon, illustration, painting, blurry, deformed, sunglasses, hat, hands, text, watermark, side view"
PRESENTER_SEEDS = [7, 11, 23, 42]

# ── quality / speed ─────────────────────────────────────────────────────────────────────
SIZE = 256                  # SadTalker face model: 256 (faster) or 512 (sharper, ~3× slower)
ENHANCER = "gfpgan"         # "gfpgan" sharpens the face (slower); "" skips it
OUTPUT_HEIGHT = 540         # height of the delivered clips; the width follows the portrait (square → 540×540)

import os, sys, json, glob, shutil, subprocess, time, pathlib, zipfile
WORK = pathlib.Path("/kaggle/working")
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"   # SadTalker's .pth.tar checkpoints predate torch's weights-only default
print("config ok")

In [ ]:
# ── environment check ───────────────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU — Settings → Accelerator → GPU T4 x2"
GPUS = torch.cuda.device_count()
print(f"torch {torch.__version__}, {GPUS} GPU(s): {[torch.cuda.get_device_name(i) for i in range(GPUS)]}")

# find the narration: a manifest.json anywhere under /kaggle/input, or a zip that contains one
manifests = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
if not manifests:
    zips = glob.glob("/kaggle/input/**/presenter-input.zip", recursive=True)
    assert zips, "Add data → upload presenter-input.zip as a Dataset and attach it to this notebook"
    with zipfile.ZipFile(zips[0]) as z: z.extractall(WORK / "input")
    manifests = glob.glob(str(WORK / "input/**/manifest.json"), recursive=True)
INPUT = pathlib.Path(manifests[0]).parent
manifest = json.loads((INPUT / "manifest.json").read_text())
episodes = manifest["episodes"] if EPISODES == "all" else [e for e in manifest["episodes"] if e["slug"] in EPISODES]
assert episodes, f"none of {EPISODES} is in the manifest; slugs: {[e['slug'] for e in manifest['episodes']]}"
print(f"{len(episodes)} episode(s), {sum(e['seconds'] for e in episodes)/60:.1f} min of narration, from {INPUT}")

In [ ]:
# ── install: SadTalker + its models, with the six fixes it needs on a 2026 stack ─────────
# Verified on CPU with torch 2.6 / numpy 2 / librosa 0.11 — the same generation Kaggle ships.
# kornia is pinned to the pure-Python 0.6 line SadTalker was written against: the newer one pulls in a
# native extension that is not built for every CPU.
%cd /kaggle/working
!pip install -q "setuptools<81" face_alignment==1.3.5 basicsr facexlib gfpgan librosa kornia==0.6.12 yacs pyyaml joblib imageio imageio-ffmpeg numba resampy pydub av safetensors huggingface_hub diffusers transformers accelerate 2>&1 | grep -vE "already satisfied|^\s*$" | tail -3
if not pathlib.Path("SadTalker").exists():
    !git clone -q --depth 1 https://github.com/OpenTalker/SadTalker.git

import importlib.util, re
def patch(path, old, new):
    p = pathlib.Path(path); s = p.read_text()
    if old in s: p.write_text(s.replace(old, new)); print("patched", p.name)

# 1. basicsr imports a torchvision module that no longer exists
bsr = pathlib.Path(importlib.util.find_spec("basicsr").submodule_search_locations[0])
patch(bsr / "data/degradations.py", "from torchvision.transforms.functional_tensor import rgb_to_grayscale", "from torchvision.transforms.functional import rgb_to_grayscale")
# 2. librosa >= 0.10 wants keyword arguments for the mel filter bank
patch("SadTalker/src/utils/audio.py", "librosa.filters.mel(hp.sample_rate, hp.n_fft, n_mels=hp.num_mels,", "librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft, n_mels=hp.num_mels,")
# 3. numpy 2 moved VisibleDeprecationWarning and dropped the np.float / np.int aliases
patch("SadTalker/src/face3d/util/preprocess.py", "np.VisibleDeprecationWarning", "np.exceptions.VisibleDeprecationWarning")
#    …and refuses float() on a one-element array, which the face-crop arithmetic relies on
patch("SadTalker/src/face3d/util/preprocess.py", "float((t[0] - w0/2)*s)", "np.ravel((t[0] - w0/2)*s)[0]")
patch("SadTalker/src/face3d/util/preprocess.py", "float((h0/2 - t[1])*s)", "np.ravel((h0/2 - t[1])*s)[0]")
patch("SadTalker/src/face3d/util/preprocess.py", "trans_params = np.array([w0, h0, s, t[0], t[1]])", "trans_params = np.array([w0, h0, s, np.ravel(t[0])[0], np.ravel(t[1])[0]], dtype=np.float64)")
patch("SadTalker/src/utils/preprocess.py", "[float(item) for item in np.hsplit(trans_params, 5)]", "[float(np.ravel(item)[0]) for item in np.hsplit(trans_params, 5)]")
# 5. seamlessClone refuses a face crop that reaches past the picture's edge; paste the visible part instead
patch("SadTalker/src/utils/paste_pic.py", "def paste_pic(", '''def _clone_or_paste(p, full_img, mask, location, box):
    """seamlessClone refuses a crop that reaches past the picture's edge; paste the visible part instead."""
    H, W = full_img.shape[:2]
    ox1, oy1, ox2, oy2 = box
    if ox1 >= 0 and oy1 >= 0 and ox2 <= W and oy2 <= H:
        return cv2.seamlessClone(p, full_img, mask, location, cv2.NORMAL_CLONE)
    cx1, cy1, cx2, cy2 = max(0, ox1), max(0, oy1), min(W, ox2), min(H, oy2)
    out = full_img.copy()
    out[cy1:cy2, cx1:cx2] = p[cy1 - oy1:cy2 - oy1, cx1 - ox1:cx2 - ox1]
    return out

def paste_pic(''')
patch("SadTalker/src/utils/paste_pic.py", "        gen_img = cv2.seamlessClone(p, full_img, mask, location, cv2.NORMAL_CLONE)", "        gen_img = _clone_or_paste(p, full_img, mask, location, (ox1, oy1, ox2, oy2))")
for f in pathlib.Path("SadTalker/src").rglob("*.py"):
    s = f.read_text(); t = re.sub(r"\bnp\.(float|int|bool)\b", r"\1", s)
    if t != s: f.write_text(t)
# 6. the .pth.tar checkpoints need torch.load(weights_only=False) — handled by the env var set in the config cell

# models: SadTalker's checkpoints from the author's Hugging Face mirror, GFPGAN's from its releases
from huggingface_hub import hf_hub_download
ck = pathlib.Path("SadTalker/checkpoints"); ck.mkdir(exist_ok=True)
for f in [f"SadTalker_V0.0.2_{SIZE}.safetensors", "mapping_00109-model.pth.tar", "mapping_00229-model.pth.tar"]:
    if not (ck / f).exists(): shutil.copy(hf_hub_download("vinthony/SadTalker-V002rc", f), ck / f)
gw = pathlib.Path("SadTalker/gfpgan/weights"); gw.mkdir(parents=True, exist_ok=True)
for url in ["https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth",
            "https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth",
            "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth",
            "https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth"]:
    if not (gw / url.rsplit("/", 1)[1]).exists(): !wget -q -O {gw / url.rsplit("/", 1)[1]} {url}
print("models:", sorted(p.name for p in ck.iterdir()), sorted(p.name for p in gw.iterdir()))

# does it import? (fails loudly here rather than an hour in)
!cd SadTalker && python -c "import sys; sys.path.insert(0,'.'); from src.utils.preprocess import CropAndExtract; from src.test_audio2coeff import Audio2Coeff; from src.facerender.animate import AnimateFromCoeff; print('SadTalker imports ok')"

In [ ]:
# ── the presenter portrait ──────────────────────────────────────────────────────────────
from PIL import Image
from IPython.display import display
OUT = WORK / "presenter"; OUT.mkdir(exist_ok=True)

if PRESENTER_IMAGE:
    portrait = Image.open(PRESENTER_IMAGE).convert("RGB")
else:
    from diffusers import AutoPipelineForText2Image
    pipe = AutoPipelineForText2Image.from_pretrained("stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16").to("cuda")
    candidates = []
    for seed in PRESENTER_SEEDS:
        g = torch.Generator("cuda").manual_seed(seed)
        img = pipe(prompt=PRESENTER_PROMPT, negative_prompt=PRESENTER_NEGATIVE, num_inference_steps=4, guidance_scale=0.0,
                   generator=g, height=512, width=512).images[0]
        img.save(OUT / f"candidate-{seed}.png"); candidates.append(img)
    grid = Image.new("RGB", (512 * len(candidates), 512))
    for i, im in enumerate(candidates): grid.paste(im, (512 * i, 0))
    grid.save(OUT / "candidates.png"); display(grid.resize((256 * len(candidates), 256)))
    print("candidates left to right = PRESENTER_PICK 0..", len(candidates) - 1, "— change PRESENTER_PICK in the config cell to use another")
    portrait = candidates[PRESENTER_PICK]
    del pipe; torch.cuda.empty_cache()

# Room around the face: SadTalker's full-frame paste needs the face crop to fit inside the picture, and
# generated portraits often sit close to an edge. Pad by 12% with the picture's own background colour.
from PIL import ImageOps
import numpy as np
edge = np.concatenate([np.array(portrait)[0], np.array(portrait)[-1], np.array(portrait)[:, 0], np.array(portrait)[:, -1]])
bg = tuple(int(v) for v in np.median(edge, axis=0))
portrait = ImageOps.expand(portrait, border=int(0.12 * portrait.width), fill=bg)
portrait.save(OUT / "presenter.png")

# exactly one face, or SadTalker will stop an hour in
import face_alignment, numpy as np
fa = face_alignment.FaceAlignment(face_alignment.LandmarksType._2D, device="cuda", flip_input=False)
faces = fa.get_landmarks(np.array(portrait))
assert faces and len(faces) == 1, f"expected one face in the portrait, found {0 if not faces else len(faces)} — pick another candidate or image"
print("presenter.png — one face detected"); display(portrait.resize((256, 256)))

In [ ]:
# ── narration → 16 kHz mono WAV (what SadTalker reads) ──────────────────────────────────
AUDIO = WORK / "audio"; AUDIO.mkdir(exist_ok=True)
for e in episodes:
    wav = AUDIO / f"{e['slug']}.wav"
    if not wav.exists():
        subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-y", "-i", str(INPUT / e["audio"]), "-ac", "1", "-ar", "16000", str(wav)], check=True)
print("audio ready:", len(list(AUDIO.glob("*.wav"))), "file(s)")

In [ ]:
# ── render: one SadTalker run per episode, two at a time (one per GPU), resumable ──────────
from concurrent.futures import ThreadPoolExecutor
LOG = WORK / "logs"; LOG.mkdir(exist_ok=True)

def render(e, gpu):
    slug = e["slug"]; final = OUT / f"{slug}.mp4"
    if final.exists():
        return slug, "already done", 0
    t0 = time.time(); result_dir = WORK / "raw" / slug; result_dir.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "inference.py", "--driven_audio", str(AUDIO / f"{slug}.wav"), "--source_image", str(OUT / "presenter.png"),
           "--result_dir", str(result_dir), "--still", "--preprocess", "full", "--size", str(SIZE), "--batch_size", "8"]
    if ENHANCER: cmd += ["--enhancer", ENHANCER]
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    with open(LOG / f"{slug}.log", "w") as log:
        r = subprocess.run(cmd, cwd="SadTalker", env=env, stdout=log, stderr=subprocess.STDOUT)
    if r.returncode:
        return slug, f"FAILED — see logs/{slug}.log", time.time() - t0
    raw = max(result_dir.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    # deliver video only (the course supplies the narration), scaled, 25 fps, streamable
    subprocess.run(["ffmpeg", "-nostdin", "-loglevel", "error", "-y", "-i", str(raw), "-an", "-vf", f"scale=-2:{OUTPUT_HEIGHT}", "-r", "25",
                    "-c:v", "libx264", "-crf", "20", "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(final)], check=True)
    return slug, "ok", time.time() - t0

with ThreadPoolExecutor(max_workers=max(1, GPUS)) as pool:
    futures = [pool.submit(render, e, i % max(1, GPUS)) for i, e in enumerate(episodes)]
    for f in futures:
        slug, status, secs = f.result()
        print(f"{slug:52} {status:28} {secs/60:5.1f} min")

In [ ]:
# ── look at one, then package everything for download ──────────────────────────────────
from IPython.display import Video
done = sorted(OUT.glob("*.mp4"))
if done: display(Video(str(done[0]), embed=False, width=360))

def seconds(p):
    return float(subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(p)], capture_output=True, text=True).stdout)
(OUT / "presenter.json").write_text(json.dumps({
    "model": f"SadTalker V0.0.2 {SIZE}" + (f" + {ENHANCER}" if ENHANCER else ""), "portrait": "presenter.png",
    "prompt": None if PRESENTER_IMAGE else PRESENTER_PROMPT, "seed": None if PRESENTER_IMAGE else PRESENTER_SEEDS[PRESENTER_PICK],
    "fps": 25, "height": OUTPUT_HEIGHT,
    "clips": {p.stem: round(seconds(p), 3) for p in done},
}, indent=1))

zip_path = WORK / "presenter-clips.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:
    for p in sorted(OUT.iterdir()):
        if p.suffix in (".mp4", ".png", ".json"): z.write(p, f"presenter/{p.name}")
print(f"{zip_path.name}: {zip_path.stat().st_size/1e6:.0f} MB, {len(done)} clip(s) — download it from the Output tab, unzip into video/ in the repo, then: npm run video")